# ESCM — baseline network

Reconstruct the original network from recorded YARP camera events. Keep the legacy neural dynamics until behavior is validated.

In [ ]:
from pathlib import Path
import numpy as np
from brian2 import *
from escm.event_input import load_events

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
recording = load_events(
    root / "InputData/MVSEC_short_outdoor/leftdvs",
    channel="left", width=346, height=260,
    timestamp_clock_ns=1000,  # Verify this for the recording
)
events = recording["events"]

## Camera spikes

The raw recording remains untouched. The legacy Brian2 simulation requires events on its 0.5 ms clock; same-pixel events on the same tick are merged.

In [ ]:
start_scope()
defaultclock.dt = 0.5 * ms
width, height = recording["width"], recording["height"]
N = width * height
tau, vt, vr = 0.2 * ms, 0.1, 0.0
beta, Wi, Wk, k = 0.5, 6.0, -3.0, 8
duration = 0.1  # seconds

ts = np.asarray(events["ts"])
if not len(ts):
    raise ValueError("Empty event stream")
use = (ts - ts[0]) < duration
pixel = np.asarray(events["y"])[use] * width + np.asarray(events["x"])[use]
ticks = np.floor((ts[use] - ts[0]) / float(defaultclock.dt / second)).astype(np.int64)
pairs = np.unique(np.column_stack((ticks, pixel)), axis=0)
camera = SpikeGeneratorGroup(N, pairs[:, 1], pairs[:, 0] * defaultclock.dt, sorted=True)

## Network

Direct full-field connectivity avoids the legacy chunk-boundary omission, so the new network is not yet an exact reproduction. Recurrent states and persistence-event logic are otherwise left as in the original implementation.

In [ ]:
equations = """
dv/dt = (-v + ExtIn + P * (Inh + Exc)) / tau : 1
dExtIn/dt = -ExtIn / tau : 1
dInh/dt = -Inh / tau : 1
dExc/dt = -Exc / tau : 1
incoming_spikes : integer
P : integer
X : integer
Y : integer
"""
neurons = NeuronGroup(
    N, equations, threshold="v > vt",
    reset="v = vr; incoming_spikes_post = 0",
    refractory=0 * ms,
    events={"P_ON": "v > vt",
            "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= vt)"},
    method="exact",
)
neurons.X = np.arange(N) % width
neurons.Y = np.arange(N) // width
neurons.set_event_schedule("P_ON", when="after_thresholds")
neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
neurons.set_event_schedule("P_OFF", when="groups")
neurons.run_on_event("P_OFF", "P = 0", when="groups")

feed = Synapses(camera, neurons, on_pre="ExtIn_post = beta")
feed.connect(j="i")
recurrent = Synapses(neurons, neurons, on_pre={
    "pre": "incoming_spikes_post += 1; Exc_pre = Wi",
    "pre_2": "Inh_post = P_post * Wk * incoming_spikes_post",
})
# All surrounding neighbors (no autapses), as in the original intended geometry.
for dy in range(-k, k + 1):
    for dx in range(-k, k + 1):
        if dx == dy == 0:
            continue
        xs = np.arange(max(0, -dx), min(width, width - dx))
        ys = np.arange(max(0, -dy), min(height, height - dy))
        if not len(xs) or not len(ys):
            continue
        xx, yy = np.meshgrid(xs, ys)
        src = (yy * width + xx).ravel()
        recurrent.connect(i=src, j=src + dy * width + dx)

## Simulate; verify output during camera silence

Continue the same network with no further camera spikes. Output during the silent interval is evidence of neuronal persistence, rather than an effect of video accumulation. This alone does not demonstrate velocity invariance.

In [ ]:
output = SpikeMonitor(neurons)
run(duration * second)
silence_duration = 0.1  # seconds
run(silence_duration * second)

output_t = np.asarray(output.t / second)
output_i = np.asarray(output.i, dtype=np.int64)
late = output_t >= duration + float(defaultclock.dt / second)
print(f"Input spikes: {len(pairs):,}")
print(f"Output spikes in silent interval: {np.count_nonzero(late):,}")
print(f"Neurons active during silence: {len(np.unique(output_i[late])):,}")

## Side-by-side video: camera events vs ESCM spikes

**No temporal accumulation or persistence is introduced by the renderer.** Every displayed image represents **exactly one 0.5 ms simulation interval**, for both sides; earlier events are not retained, and neither panel is interpolated across frames. Playback at 60 FPS is slow motion (~33× slower than real time), preserving the original event-level temporal window irrespective of scene velocity. The labeled camera-silence period provides an immediate check that persistent ESCM output comes from network activity. The camera panel shows raw spikes delivered to the model, not an intensity-frame ground truth.

In [ ]:
import imageio.v2 as imageio
from PIL import Image, ImageDraw

fps = 60  # playback only; does not set the event integration interval
video_path = root / "escm_input_vs_output.mp4"
dt_s = float(defaultclock.dt / second)
input_t = pairs[:, 0] * dt_s
input_i = pairs[:, 1].astype(np.int64)
n_frames = int(np.ceil((duration + silence_duration) / dt_s))

with imageio.get_writer(video_path, fps=fps, codec="libx264", macro_block_size=1) as writer:
    for frame_id in range(n_frames):
        t0 = frame_id * dt_s
        t1 = (frame_id + 1) * dt_s
        a, b = np.searchsorted(input_t, [t0, t1], side="left")
        c, d = np.searchsorted(output_t, [t0, t1], side="left")
        input_frame = np.zeros(N, dtype=np.uint8)
        output_frame = np.zeros(N, dtype=np.uint8)
        input_frame[input_i[a:b]] = 255
        output_frame[output_i[c:d]] = 255

        frame = np.zeros((height + 44, 2 * width, 3), dtype=np.uint8)
        frame[44:, :width] = input_frame.reshape(height, width)[:, :, None]
        frame[44:, width:] = output_frame.reshape(height, width)[:, :, None]
        image = Image.fromarray(frame)
        draw = ImageDraw.Draw(image)
        draw.text((6, 4), "Camera events (0.5 ms)", fill="white")
        draw.text((width + 6, 4), "ESCM spikes (0.5 ms)", fill="white")
        draw.text((6, 23), f"t = {t0:.4f} s", fill="white")
        if t0 >= duration:
            draw.text((width + 6, 23), "NO CAMERA INPUT", fill="white")
        writer.append_data(np.asarray(image))

print("Saved:", video_path)